# XGBoost

Standard tabular ML workflow for LD50 regression.


In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np

PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from sklearn.linear_model import RidgeCV

from utils.modeling import artifact_paths, load_tabular_arrays, save_ml_run

In [2]:
MODEL_NAME = "xgboost"
BASE_DIR = Path.cwd()
RANDOM_STATE = 42

X_train, X_valid, X_test, y_train, y_valid, y_test, feature_names = load_tabular_arrays('../../data/feature_selection')
X_train = X_train.astype(np.float32)
X_valid = X_valid.astype(np.float32)
X_test = X_test.astype(np.float32)

paths = artifact_paths(
    BASE_DIR,
    MODEL_NAME,
    model_suffix=".pkl",
    include_importance=True,
)


In [3]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    device="cuda",
    random_state=RANDOM_STATE,
)

try:
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
except Exception as exc:
    warnings.warn(f"XGBoost CUDA failed; retrying on CPU. Details: {exc}")
    model.set_params(device="cpu")
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

predictions = model.predict(X_test)

c:\Users\Tommaso\miniconda3\envs\hack_03\Lib\site-packages\xgboost\core.py:751: UserWarning: [15:23:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [4]:
metrics = save_ml_run("XGBoost", model, paths, X_test, y_test, predictions, feature_names)

[XGBoost] RMSE: 0.5936 | MAE: 0.4347 | R2: 0.6056
